<a href="https://colab.research.google.com/github/Ashuu9589/ashish-portfolio/blob/main/Analyzing_Historical_Stock_Revenue_Data_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on Lab: Analyzing Historical Stock/Revenue Data and Building a Dashboard

In this notebook we will:
- Extract Tesla and GameStop stock data using `yfinance`
- Extract Tesla and GameStop revenue data using web scraping (`requests` + `BeautifulSoup`)
- Build interactive dashboards comparing stock price and revenue over time

## Setup: Import Libraries

In [10]:
!pip install yfinance==0.2.28 --quiet
!pip install bs4==0.0.1 --quiet
!pip install pandas==1.4.4 --quiet
!pip install plotly==5.11.0 --quiet
!pip install lxml --quiet

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [11]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

## Define Graphing Function

This function will be reused for both the Tesla and GameStop dashboards.

In [12]:
def make_graph(stock_data, revenue_data, stock):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                         subplot_titles=("Historical Share Price", "Historical Revenue"),
                         vertical_spacing=.3)

    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']

    fig.add_trace(go.Scatter(x=pd.to_datetime(stock_data_specific.Date),
                              y=stock_data_specific.Close.astype("float"),
                              name="Share Price"), row=1, col=1)

    fig.add_trace(go.Scatter(x=pd.to_datetime(revenue_data_specific.Date),
                              y=revenue_data_specific.Revenue.astype("float"),
                              name="Revenue"), row=2, col=1)

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)

    fig.update_layout(showlegend=False,
                       height=900,
                       title=stock,
                       xaxis_rangeslider_visible=True)
    fig.show()

## Question 1: Extract Tesla Stock Data Using `yfinance`

Use the `Ticker` function to create a ticker object for Tesla with the ticker symbol **TSLA**. Use the function `history` to extract stock information and save it as a dataframe named `tesla_data`. Reset the index, then display the first five rows using the `head` function.

In [13]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

ERROR:yfinance:Failed to get ticker 'TSLA' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:TSLA: No timezone found, symbol may be delisted


,Date,Open,High,Low,Close,Adj Close,Volume


## Question 2: Extract Tesla Revenue Data Using Webscraping

Use `requests` to download the webpage containing Tesla's quarterly revenue data, then parse it with `BeautifulSoup`. Store the result as a dataframe named `tesla_revenue` with columns `Date` and `Revenue`, remove the comma and dollar sign from the `Revenue` column, and drop any null/empty rows.

In [14]:
url = "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue"
headers = {"User-Agent": "Mozilla/5.0"}
html_data = requests.get(url, headers=headers).text

soup = BeautifulSoup(html_data, "html5lib")

tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])

for table in soup.find_all("table"):
    if "Tesla Quarterly Revenue" in str(table.find_previous("h3")) or True:
        for row in table.find("tbody").find_all("tr"):
            col = row.find_all("td")
            if len(col) != 2:
                continue
            date = col[0].text
            revenue = col[1].text
            tesla_revenue = pd.concat(
                [tesla_revenue, pd.DataFrame({"Date": [date], "Revenue": [revenue]})],
                ignore_index=True
            )
        break

tesla_revenue["Revenue"] = tesla_revenue["Revenue"].str.replace(",", "").str.replace("$", "", regex=False)
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue["Revenue"] != ""]

tesla_revenue.tail()

,Date,Revenue


## Question 3: Extract GameStop Stock Data Using `yfinance`

Use the `Ticker` function to create a ticker object for GameStop with the ticker symbol **GME**. Extract stock information, reset the index, then display the first five rows.

In [15]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

ERROR:yfinance:Failed to get ticker 'GME' reason: Expecting value: line 1 column 1 (char 0)
ERROR:yfinance:GME: No timezone found, symbol may be delisted


,Date,Open,High,Low,Close,Adj Close,Volume


## Question 4: Extract GameStop Revenue Data Using Webscraping

Extract GameStop's quarterly revenue data the same way, saving it as `gme_revenue` with `Date` and `Revenue` columns, cleaned of `$` and `,` characters.

In [16]:
url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"
headers = {"User-Agent": "Mozilla/5.0"}
html_data = requests.get(url, headers=headers).text

soup = BeautifulSoup(html_data, "html5lib")

gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])

for table in soup.find_all("table"):
    for row in table.find("tbody").find_all("tr"):
        col = row.find_all("td")
        if len(col) != 2:
            continue
        date = col[0].text
        revenue = col[1].text
        gme_revenue = pd.concat(
            [gme_revenue, pd.DataFrame({"Date": [date], "Revenue": [revenue]})],
            ignore_index=True
        )
    break

gme_revenue["Revenue"] = gme_revenue["Revenue"].str.replace(",", "").str.replace("$", "", regex=False)
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue["Revenue"] != ""]

gme_revenue.tail()

,Date,Revenue


## Question 5: Tesla Stock and Revenue Dashboard

Use the `make_graph` function to display both the Tesla stock price graph and revenue graph.

In [17]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

## Question 6: GameStop Stock and Revenue Dashboard

Use the `make_graph` function to display both the GameStop stock price graph and revenue graph.

In [18]:
make_graph(gme_data, gme_revenue, 'GameStop')

## Quiz Notes (for the assignment's short-answer questions)

- **Q2 (parsers in BeautifulSoup):** common values are `"html.parser"`, `"lxml"`, and `"html5lib"`.
- **Q3 (why convert `Date` to datetime):** it lets pandas/plotly sort chronologically, filter by date range, and correctly plot a continuous time series instead of treating dates as text.
- **Q4 (removing `$` and `,`):** use `.str.replace(',', '').str.replace('$', '')` on the column before converting to float.
- **Q5/Q6 (specific values):** run the notebook top-to-bottom in your own environment (Coursera/Skills Network Lab) and read the exact last-row `Revenue` value from `tesla_revenue.tail()` and the first-row `Close` value from `gme_data.head()` (Date 2002-02-13) — live scraped/pulled values can shift slightly by run date, so the grader wants the number you actually got.
- **Q7 (Tesla trend 2018–2021):** stock price stayed relatively flat/low through 2019 then rose sharply through 2020–2021, while revenue grew steadily throughout — price appreciation ran well ahead of revenue growth.
- **Q8 (GameStop):** revenue trended gradually downward over the period, while the stock price stayed low and flat for most of it, then spiked dramatically in early 2021 — a sharp disconnect between price and underlying revenue.